<a target="_blank" href="https://colab.research.google.com/github/PacktPublishing/Building-Agentic-AI-Systems/blob/main/Chapter_06.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

# Chapter 6 – Exploring the Coordinator, Worker, and Delegator Approach
---

Install dependencies

In [1]:
!pip install crewai langchain-openai

In [2]:
import getpass
import os

api_key = getpass.getpass(prompt="Enter OpenAI API Key: ")
os.environ["OPENAI_API_KEY"] = api_key

Enter OpenAI API Key:  ········


### Role-based agents

Role-based agents within the CWD (Coordinator, Worker, and Delegator) model for a travel planner.

# CrewAI implementation
---

In [3]:
from crewai import Agent, Task, Crew, Process
from crewai.tools import tool
from IPython.display import display, Markdown, HTML

llm = "gpt-4o"

<div class="alert alert-block alert-info"> 
<b>NOTE:</b> While we will use gpt-4o throughout this notebook, you can also use different LLMs for each of the agents. This is usually a recommended approach. For example for less complex tasks such as crafting a plan to book a travel itinerary, one could use a smaller model such as gpt-4o-mini, and for more complex tasks such as comparing travel options and reasoning a larger model is appropriate.
</div>


## Create the tools

In [4]:
@tool("Search for available flights between cities")
def search_flights(origin: str, destination: str, date: str) -> dict:
    """
    Search for available flights between cities.
    
    Args:
        origin: Departure city
        destination: Arrival city
    
    Returns:
        Dictionary containing flight options and prices
    """
    # Emulate JSON data from an API
    return {
        "flights": [
            {"airline": "Air France", "price": 850, "departure": "New York (JFK)", "arrival": "Paris (CDG)", "duration": "7h 30m", "departure_time": "10:30 AM", "arrival_time": "11:00 PM"},
            {"airline": "Delta Airlines", "price": 780, "departure": "New York (JFK)", "arrival": "Paris (CDG)", "duration": "7h 45m", "departure_time": "5:30 PM", "arrival_time": "6:15 AM"},
            {"airline": "United Airlines", "price": 920, "departure": "New York (EWR)", "arrival": "Paris (CDG)", "duration": "7h 55m", "departure_time": "8:45 PM", "arrival_time": "9:40 AM"}
        ]}             

@tool("Find available hotels in a location") 
def find_hotels(location: str, check_in: str, check_out: str) -> dict:
    """
    Search for available hotels in a location.
    
    Args:
        location: City name
        check_in: Check-in date (YYYY-MM-DD)
        check_out: Check-out date (YYYY-MM-DD)
    
    Returns:
        Dictionary containing hotel options and prices
    """
    # Emulate JSON data from an API
    return {
        "hotels": [
            {"name": "Paris Marriott Champs Elysees", "price": 450, "check_in_date": check_in, "check_out_date": check_out, "rating": 4.5, "location": "Central Paris", "amenities": ["Spa", "Restaurant", "Room Service"]},
            {"name": "Citadines Saint-Germain-des-Prés", "price": 320, "check_in_date": check_in, "check_out_date": check_out, "rating": 4.2, "location": "Saint-Germain", "amenities": ["Kitchenette", "Laundry", "Wifi"]},
            {"name": "Ibis Paris Eiffel Tower", "price": 380, "check_in_date": check_in, "check_out_date": check_out, "rating": 4.0, "location": "Near Eiffel Tower", "amenities": ["Restaurant", "Bar", "Wifi"]}
        ]}

@tool("Find available activities in a location")
def find_activities(location: str, date: str, preferences: str) -> dict:
    """
    Find available activities in a location.
    
    Args:
        location: City name
        date: Activity date (YYYY-MM-DD)
        preferences: Activity preferences/requirements
        
    Returns:
        Dictionary containing activity options
    """
    # Implement actual activity search logic here
    return {
        "activities": [
            {"name": "Eiffel Tower Skip-the-Line", "description": "Priority access to the Eiffel Tower with guided tour of 1st and 2nd floors", "price": 65, "duration": "2 hours", "start_time": "10:00 AM", "meeting_point": "Eiffel Tower South Entrance"},
            {"name": "Louvre Museum Guided Tour", "description": "Expert-guided tour of the Louvre's masterpieces including Mona Lisa", "price": 85, "duration": "3 hours", "start_time": "2:00 PM", "meeting_point": "Louvre Pyramid"},
            {"name": "Seine River Dinner Cruise", "description": "Evening cruise along the Seine with 3-course French dinner and wine", "price": 120, "duration": "2.5 hours", "start_time": "7:30 PM", "meeting_point": "Port de la Bourdonnais"}
        ]}

@tool("Find local transportation options")
def find_transportation(location: str, origin: str, destination: str) -> dict:
    """
    Find local transportation options between locations.
    
    Args:
        location: City name
        origin: Starting point (e.g., "Airport", "Hotel", "Eiffel Tower")
        destination: End point (e.g., "City Center", "Museum", "Restaurant")
    
    Returns:
        Dictionary containing transportation options
    """
    # Return a simple JSON with transportation options
    return {
        "options": [
            { "type": "Metro", "cost": 1.90, "duration": "25 minutes", "frequency": "Every 5 minutes", "route": "Line 1 to Châtelet, then Line 4 to destination", "pros": "Fast, avoids traffic", "cons": "Can be crowded during peak hours"},
            { "type": "Taxi", "cost": 22.50, "duration": "20 minutes", "frequency": "On demand", "route": "Direct", "pros": "Door-to-door service, comfortable", "cons": "More expensive, subject to traffic"},
            { "type": "Bus", "cost": 1.90, "duration": "35 minutes", "frequency": "Every 10 minutes", "route": "Route 42 direct to destination", "pros": "Scenic route, above ground", "cons": "Slower than metro, subject to traffic"},
            { "type": "Walking", "cost": 0, "duration": "45 minutes", "frequency": "Anytime", "route": "Through city center", "pros": "Free, healthy, scenic", "cons": "Takes longer, weather dependent"}
        ],
        "passes": [
            { "name": "Day Pass", "cost": 7.50, "valid_for": "Unlimited travel for 24 hours", "recommended_if": "Making more than 4 trips in a day" },
            { "name": "Paris Visite",  "cost": 12.00, "valid_for": "Unlimited travel for 1 day, includes discounts to attractions", "recommended_if": "Planning to visit multiple tourist sites" }
        ]
    }

## Create the Agents

### Core Travel Workers

In [5]:
flight_booking_worker = Agent(
    role="Flight Booking Specialist",
    goal="Find and book the optimal flights for the traveler",
    backstory="""You are an experienced flight booking specialist with extensive knowledge of airlines, 
    routes, and pricing strategies. You excel at finding the best flight options balancing cost, 
    convenience, and comfort according to the traveler's preferences.""",
    verbose=True,
    allow_delegation=False,
    tools=[search_flights],
    llm=llm,
    max_iter=1,
    max_retry_limit=3
)

hotel_booking_worker = Agent(
    role="Hotel Accommodation Expert",
    goal="Secure the ideal hotel accommodations for the traveler",
    backstory="""You have worked in the hospitality industry for over a decade and have deep knowledge 
    of hotel chains, boutique accommodations, and local lodging options worldwide. You're skilled at 
    matching travelers with accommodations that meet their budget, location preferences, and amenity requirements.""",
    verbose=True,
    allow_delegation=False,
    tools=[find_hotels],
    llm=llm,
    max_iter=1,
    max_retry_limit=3
)

activity_planning_worker = Agent(
    role="Activities and Excursions Planner",
    goal="Curate personalized activities and experiences for the traveler",
    backstory="""You're a well-traveled activities coordinator with insider knowledge of attractions, 
    tours, and unique experiences across numerous destinations. You're passionate about creating 
    memorable itineraries that align with travelers' interests, whether they seek adventure, culture, 
    relaxation, or culinary experiences.""",
    verbose=True,
    allow_delegation=False,
    tools=[find_activities],
    llm=llm,
    max_iter=1,
    max_retry_limit=3
)

transportation_worker = Agent(
    role="Local Transportation Coordinator",
    goal="Arrange efficient and convenient local transportation",
    backstory="""You specialize in local transportation logistics across global destinations. Your expertise 
    covers public transit systems, private transfers, rental services, and navigation, ensuring travelers 
    can move smoothly between destinations and activities.""",
    verbose=True,
    allow_delegation=False,
    tools=[find_transportation],
    llm=llm,
    max_iter=1,
    max_retry_limit=3
)

## Define tasks for all the CWD agents

### Tasks for the workers

In [6]:
flight_search_task = Task(
    description="""
    Use the search_flights tool to find flight options from origin to destination.
    Review the returned JSON data and recommend the best option based on the traveler's priorities, if any.
    
    Compare the available options and recommended choice best meets their needs.
    """,
    agent=flight_booking_worker,
    expected_output="A flight itinerary for booking based on the traveler's preferences."
)

hotel_search_task = Task(
    description="""
    Use the find_hotels tool to search for accommodations in the destination.
    Review the returned JSON data and recommend the best option considering budget.
    
    Explain why your recommended choice is the best match for this traveler.
    """,
    agent=hotel_booking_worker,
    expected_output="A hotel recommendation based on the traveler's preferences and budget."
)

activity_planning_task = Task(
    description="""
    Use the find_activities tool to identify options in the destination for each day of the of the entire trip duration.
    The traveler's interests are: {activity_interests} with a {activity_pace} pace preference.
    
    Create a day-by-day plan using the returned JSON data, ensuring activities flow logically and match the traveler's interests.
    """,
    agent=activity_planning_worker,
    expected_output="A day-by-day activity plan that matches the traveler's interests and pace preferences."
)

transportation_planning_task = Task(
    description="""
    Use the find_transportation tool to identify options at the destination for:
    1. Airport to hotel transfer
    2. Transportation between daily activities
    3. Hotel to airport transfer
    
    Consider the traveler's preferences where possible.
    
    Based on the returned JSON data, recommend the best transportation options for each segment of their trip.
    """,
    agent=transportation_worker,
    expected_output="A transportation plan covering all necessary transfers during the trip."
)

### Defining the Coordinator Agent & Task

The `coordinate_request` function will use our Coordinator agent with a task to consume customer requests and craft a "plan" for the delegator agent later.

In [7]:

coordinator_agent = Agent(
    role="Coordinator Agent",
    goal="Ensure cohesive travel plans and maintain high customer satisfaction",
    backstory="""A seasoned travel industry veteran with 15 years of experience in luxury travel planning 
    and project management. Known for orchestrating seamless multi-destination trips for high-profile clients 
    and managing complex itineraries across different time zones and cultures. 
    """,
    verbose=False,
    llm=llm,
    max_iter=1,
    max_retry_limit=3
)

In [8]:
from textwrap import dedent

async def coordinate_request(traveler_request):

    coordinator_to_delegator_task = Task(
        description=dedent(f"""\
        As the Coordinator Agent, you've received a travel planning request.
        
        Traveler request:
        {traveler_request}
        
        Create a clear, concise travel planning steps for this trip. Only plan
        for the things requested by the traveler, DO NOT assume or add things not requested. Provide a 
        short overview, followed by the steps required for flight booking, hotel booking, activities,
        and local transportation.
        
        Your output should be a step-by-step plan along with preference details that the Delegator Agent 
        can use to effectively assign tasks to the specialist workers. Do not provide any summary or mention 
        "Delegator" or "coordinator".
        """),
        expected_output="A detailed step-by-step travel plan for the delegator agent",
        agent=coordinator_agent
    )

    # Execute the coordinator's initial planning task
    coordinator_crew = Crew(
        agents=[coordinator_agent],
        tasks=[coordinator_to_delegator_task],
        verbose=False, # True if you want to see detailed execution
        process=Process.sequential
    )
    coordinator_plan = await coordinator_crew.kickoff_async(inputs={'traveler_request': traveler_request})
    print("\n=== Coordinator Planning Complete ===\n")    
    return coordinator_plan


Test to see if our coordinator agent is creating a detailed plan for the delegator agent.

In [11]:
request="""Traveler Alex Johnson is planning to travel to Paris from New York for his anniversary for 7 days and 2 people. 
- His total budget is about $8000, with hotel budget being $300.
- Direct flights preferred, morning departure if possible.
- Hotel in Paris under $400 with wifi preferred. Check in at 5/7/2025 and checkout at 5/14/2025
- Activities in paris should be moderate pace with some relaxation time built in
- Mix of walking and public transit, with occasional taxis for evening outings
"""
plan_for_delegator = await coordinate_request(request)


=== Coordinator Planning Complete ===



View the plan crafted by the coordinator agent.

In [12]:
display(HTML('<div style="background-color: #000; padding: 10px; border-radius: 5px; border: 1px solid #d3d3d3;"></hr><h2>🔽 &nbsp; Full step-by-step trip plan</h2></hr></div>'))
display(Markdown(plan_for_delegator.raw))

**Travel Planning Steps for Alex Johnson's Trip to Paris**

**Overview:**

Alex Johnson is planning an anniversary trip to Paris from New York for two people over 7 days with a total budget of $8000. The hotel budget is about $300 per night. The trip prioritizes direct flights with a morning departure, a hotel with Wi-Fi under $400 per night, and a balanced itinerary with moderate-paced activities, relaxation, walking, and public transit, with occasional taxi use for evening outings.

---

**Step-by-Step Plan:**

**1. Flight Booking:**

   - **Preference Details:**
     - Departure from New York to Paris
     - Direct flights preferred
     - Morning departure timing if possible

   - **Steps:**
     - Search for airlines offering direct flights from New York (JFK or Newark) to Paris (Charles de Gaulle or Orly).
     - Prioritize flights departing in the morning to align with traveler's preference.
     - Compare ticket prices to ensure they fit within the broader $8000 total budget.
     - Book and confirm tickets once a suitable flight is found.

**2. Hotel Booking:**

   - **Preference Details:**
     - Check-in on 5/7/2025 and checkout on 5/14/2025
     - Budget for hotel: under $400 per night
     - Must have Wi-Fi available

   - **Steps:**
     - Search for hotels in central Paris that offer rooms under $400 per night with Wi-Fi.
     - Verify the availability for the required dates and any ongoing promotions.
     - Consider hotel reviews and location convenience for access to public transit.
     - Reserve the room ensuring the booking is flexible (if needed).

**3. Activities Planning:**

   - **Preference Details:**
     - Activities at a moderate pace with built-in relaxation time
     - Mixed use of walking and public transit, taxis for evening outings

   - **Steps:**
     - Research popular activities and attractions in Paris suitable for a moderate pace, such as visits to museums, strolling in parks, and scheduled relaxation periods.
     - Schedule visits to major sights such as the Eiffel Tower, Louvre Museum, and Notre-Dame Cathedral while considering public transit routes.
     - Plan for one or two evenings with pre-booked shows or dinner reservations requiring taxi services.
     - Include one or two days with lighter schedules to allow for leisure and relaxation.

**4. Local Transportation:**

   - **Preference Details:**
     - Predominantly use of public transit and walking
     - Occasional use of taxis, especially for evening outings

   - **Steps:**
     - Purchase a weekly public transit pass (e.g., Navigo Découverte pass) covering zones needed for tourist attractions.
     - Research and note reliable taxi services or ride-share apps (like Uber) available in Paris.
     - Prepare a list of major subway and bus routes connecting the hotel to key attractions.
     - Arrange for taxi bookings for planned evening outings and confirm with local providers.

By following these steps, the travel arrangements will align with Alex's preferences, ensuring a memorable and enjoyable anniversary trip to Paris.

### Defining the Delegator Agent & Task

The `delegate_plan` function will use the travel plan crafted by the coordinator agent and subsequently delegate tasks to worker agents for each task (such book flight, book hotel etc.). It will also subsequently process the outputs of each worker agent and then craft a full itinerary for the traveler. Here, we use the `plan` generated by the `coordinator_agent` to craft a `goal` for the `delegator_agent`. 

We will use CrewAI's `manager_agent` feature to implement Delegator, which will manage the worker agents to search flights, search hotels, plan activities and look for local transportation using the respective worker agent.

In [17]:
async def delegate_plan(plan):
    delegator_goal=f"""
        Effectively distribute travel planning tasks to specialized workers to create a detailed booking itinerary
        for the plan below:
        
        {plan}
        
        Based on this plan, your goal is to create a detailed booking itinerary and trip plan for the user that includes
        flight booking & cost recommendation, hotels and hotel cost, activities and local transportation options
        and recommendations.
        """

    delegator_agent = Agent(
        role="Travel Planning Delegator",
        goal=delegator_goal,
        backstory="""You are an expert project manager with a talent for breaking down travel planning into 
        component tasks and assigning them to the right specialists. You understand each worker's strengths 
        and ensure they have the information needed to excel. You track progress, resolve bottlenecks, and 
        ensure all elements of the trip are properly addressed.""",    
        verbose=True,
        allow_delegation=True,
        llm=llm
    )
    # Execute the delegator's task assignment
    delegator_crew = Crew(
        agents=[flight_booking_worker, hotel_booking_worker, transportation_worker, activity_planning_worker],
        tasks=[flight_search_task, hotel_search_task, transportation_planning_task, activity_planning_task ],
        verbose=False,
        manager_agent=delegator_agent,
        process=Process.hierarchical,
        planning=True,        
        full_output=True
    )
    full_itinerary = await delegator_crew.kickoff_async()
    print("\n=== Delegator Task Complete ===\n")
    return full_itinerary

<div class="alert alert-block alert-info"> 
<b>NOTE:</b> When you execute the following code cell you will see the full verbose execution of the Multi-agent delegator agent. You may also notice that at certain points the delegator failed to invoke the tool. This happens in case the LLM was unable to capture the required variables for the tool, at which point the CrewAI framework will retry the call by re-crafting it's inputs until it gets a proper tool call (often with smaller or cheaper LLMs). This is unfortunately one of the drawbacks of generic implementations, however with more custom implementations with CrewAI, you can steer the model to generate appropriate tool calls everytime given all the information is present.<br/>

Also note that the max_iter and max_retries_limit is set to 1 and 3 which means the agent will only be invoked once and will retry 3 times if there are errors. This means that the Agent may not come to a perfect answer with just 1 try, you may try to increast max_iter on the agents to experiment with the type of answers it produces.
</div>


In [18]:
itinerary = await delegate_plan(plan_for_delegator.raw)

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Delegator                                                                               │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Use the search_flights tool to find flight options from origin to destination.                             │
│      Review the returned JSON data and recommend the best option based on the traveler's priorities, if any.    │
│                                                                                                                 │
│      Compare the available options and recommended choice best meets their needs.                               │
│      1. Identify the origin and destination from the traveler’s request before invoking the tool.               │
│  2. Call the Search for available flights between cities tool with the exact origin and destination values.     │
│  3. Inspect the returned JSON carefully and extract all available flight options, including price, schedule,    │
│  duration, stops, airline, and any notable restrictions.                                                        │
│  4. Determine the traveler’s priorities if they are available in the request, such as lowest price, shortest    │
│  duration, fewer stops, preferred departure time, or airline preference.                                        │
│  5. Compare all returned options against those priorities, ranking the flights from best fit to least fit.      │
│  6. Select the single best option that most closely matches the traveler’s needs while still being practical    │
│  and reasonable.                                                                                                │
│  7. If multiple options are close, explain the tradeoffs clearly using the JSON data, such as a slightly        │
│  higher price for a much shorter travel time or a nonstop flight over a connecting itinerary.                   │
│  8. Present the recommended flight as a booking-ready itinerary, including the key details needed for a final   │
│  decision.                                                                                                      │
│  9. Ensure the final recommendation explicitly states why this option is the best match compared with the       │
│  alternatives.                                                                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool search_for_available_flights_between_cities executed with result: {'flights': [{'airline': 'Air France', 'price': 850, 'departure': 'New York (JFK)', 'arrival': 'Paris (CDG)', 'duration': '7h 30m', 'departure_time': '10:30 AM', 'arrival_time': '11:00 PM'}, {'airline...
[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Delegator                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  After reviewing the available flight options from New York to Paris on May 7, 2025, here are the best choices  │
│  based on Alex Johnson's preferences for morning departures, cost, and duration:                                │
│                                                                                                                 │
│  1. **Air France**                                                                                              │
│     - **Departure:** New York (JFK) at 10:30 AM                                                                 │
│     - **Arrival:** Paris (CDG) at 11:00 PM                                                                      │
│     - **Duration:** 7h 30m                                                                                      │
│     - **Price:** $850                                                                                           │
│     - **Notes:** This flight meets the preference for a morning departure and offers a relatively short         │
│  duration at a reasonable price.                                                                                │
│                                                                                                                 │
│  2. **Delta Airlines**                                                                                          │
│     - **Departure:** New York (JFK) at 5:30 PM                                                                  │
│     - **Arrival:** Paris (CDG) at 6:15 AM (next day)                                                            │
│     - **Duration:** 7h 45m                                                                                      │
│     - **Price:** $780                                                                                           │
│     - **Notes:** While this flight is less expensive, it departs in the evening, which does not align with the  │
│  preference for a morning departure.                                                                            │
│                                                                                                                 │
│  3. **United Airlines**                                                                                         │
│     - **Departure:** New York (EWR) at 8:45 PM                                                                  │
│     - **Arrival:** Paris (CDG) at 9:40 AM (next day)                                                            │
│     - **Duration:** 7h 55m                                                                                      │
│     - **Price:** $920                                                                                           │
│     - **Notes:** This flight is the most expensive and departs at night, which is not ideal for the morning     │
│  departure preference.                                                                                          │
│                                                                                                                 │
│  **Recommendation:**                                                                                            │
│                                                                                                                 │
│  The best option is the **Air France** flight. It align

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Delegator                                                                               │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Use the find_hotels tool to search for accommodations in the destination.                                  │
│      Review the returned JSON data and recommend the best option considering budget.                            │
│                                                                                                                 │
│      Explain why your recommended choice is the best match for this traveler.                                   │
│      1. Confirm the destination city and any available stay dates before using the tool.                        │
│  2. Call the Find available hotels in a location tool with the destination and the correct check-in and         │
│  check-out dates in YYYY-MM-DD format.                                                                          │
│  3. Review the JSON response in full and identify all hotel options, focusing on price, room type, rating,      │
│  amenities, and location if included.                                                                           │
│  4. Compare the listed hotels with the traveler’s budget emphasis, prioritizing affordability while still       │
│  considering comfort and convenience.                                                                           │
│  5. Eliminate options that are clearly outside the budget or offer poor value relative to the alternatives.     │
│  6. Choose the hotel that gives the best overall budget fit, balancing cost with useful amenities and           │
│  acceptable quality.                                                                                            │
│  7. If there are multiple low-cost options, compare value indicators such as location proximity, included       │
│  amenities, and rating to justify the final recommendation.                                                     │
│  8. Explain clearly why the chosen hotel is the best match for this traveler’s budget, using the returned data  │
│  as evidence.                                                                                                   │
│  9. Provide the recommendation in a hotel-booking-ready format with the most important details surfaced for     │
│  quick decision-making.                                                                                         │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool find_available_hotels_in_a_location executed with result: {'hotels': [{'name': 'Paris Marriott Champs Elysees', 'price': 450, 'check_in_date': '2025-05-07', 'check_out_date': '2025-05-14', 'rating': 4.5, 'location': 'Central Paris', 'amenities': ['Spa', 'Res...
[Finalize] todos_count=0, todos_with_results=0


...

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Delegator                                                                               │
│                                                                                                                 │
│  Task:                                                                                                          │
│      Use the find_activities tool to identify options in the destination for each day of the of the entire      │
│  trip duration.                                                                                                 │
│      The traveler's interests are: {activity_interests} with a {activity_pace} pace preference.                 │
│                                                                                                                 │
│      Create a day-by-day plan using the returned JSON data, ensuring activities flow logically and match the    │
│  traveler's interests.                                                                                          │
│      1. Identify the destination, trip duration, traveler interests, and pace preference from the task context  │
│  before starting the activity search.                                                                           │
│  2. Break the full trip into individual days so each day can be planned sequentially and logically.             │
│  3. For each day, call the Find available activities in a location tool with the destination, the specific      │
│  date, and the traveler’s preferences.                                                                          │
│  4. Review each JSON response carefully and collect the available activities, including category, timing,       │
│  intensity, duration, cost, and any special requirements if provided.                                           │
│  5. Match activities to the traveler’s interests, selecting experiences that directly reflect the stated        │
│  preferences such as cultural, outdoor, culinary, adventure, relaxation, or family-friendly options.            │
│  6. Respect the traveler’s pace preference by pacing the itinerary appropriately, using a relaxed structure     │
│  for slow pace, a balanced spread for moderate pace, or a fuller schedule for fast pace.                        │
│  7. Arrange activities so they flow logically from one day to the next, avoiding unnecessary backtracking or    │
│  conflicting timing patterns.                                                                                   │
│  8. If the JSON offers multiple viable options for a day, prioritize the one that best complements the rest of  │
│  the trip and creates a coherent day-by-day experience.                                                         │
│  9. Build a complete daily plan that clearly labels each day, lists the selected activity or activities, and    │
│  briefly explains how they satisfy the traveler’s interests and pace.                                           │
│  10. Ensure the final itinerary feels polished, realistic, and tailored, using only the returned data to        │
│  support each recommendation.                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Tool find_available_activities_in_a_location executed with result: {'activities': [{'name': 'Eiffel Tower Skip-the-Line', 'description': 'Priority access to the Eiffel Tower with guided tour of 1st and 2nd floors', 'price': 65, 'duration': '2 hours', 'start_time': '1...
Tool find_available_activities_in_a_location executed with result: {'activities': [{'name': 'Eiffel Tower Skip-the-Line', 'description': 'Priority access to the Eiffel Tower with guided tour of 1st and 2nd floors', 'price': 65, 'duration': '2 hours', 'start_time': '1...
Tool find_available_activities_in_a_location executed with result: {'activities': [{'name': 'Eiffel Tower Skip-the-Line', 'description': 'Priority access to the Eiffel Tower with guided tour of 1st and 2nd floors', 'price': 65, 'duration': '2 hours', 'start_time': '1...
Tool find_available_activities_in_a_location executed with result: {'activities': [{'name': 'Eiffel Tower Skip-the-Line', 'description': 'Priority access to the Eiffel Tower with guided t

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Travel Planning Delegator                                                                               │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  Here's a carefully curated day-by-day activity plan for Alex Johnson's anniversary trip to Paris. The          │
│  itinerary includes cultural and relaxing experiences suited for a moderate pace, matching their interests:     │
│                                                                                                                 │
│  ### Day 1: Arrival and Evening Cruise (May 7, 2025)                                                            │
│  - **Activity:** Seine River Dinner Cruise                                                                      │
│    - **Description:** Evening cruise along the Seine with a 3-course French dinner and wine.                    │
│    - **Start Time:** 7:30 PM                                                                                    │
│    - **Duration:** 2.5 hours                                                                                    │
│    - **Meeting Point:** Port de la Bourdonnais                                                                  │
│    - **Cost:** $120 per person                                                                                  │
│                                                                                                                 │
│  **Flow:** After arriving and settling into the hotel, this relaxing dinner cruise will provide a delightful    │
│  introduction to Paris and its iconic landmarks beautifully lit at night.                                       │
│                                                                                                                 │
│  ### Day 2: Iconic Paris Landmarks (May 8, 2025)                                                                │
│  - **Morning Activity:** Eiffel Tower Skip-the-Line Tour                                                        │
│    - **Description:** Priority access to the Eiffel Tower with a guided tour of the 1st and 2nd floors.         │
│    - **Start Time:** 10:00 AM                                                                                   │
│    - **Duration:** 2 hours                                                                                      │
│    - **Meeting Point:** Eiffel Tower South Entrance                                                             │
│    - **Cost:** $65 per person                                                                                   │
│                                                                                                                 │
│  - **Afternoon Activity:** Louvre Museum Guided Tour                                                            │
│    - **Description:** Expert-guided tour of the Louvre's masterpieces including Mona Lisa.                      │
│    - **Start Time:** 2:00 PM                                                                                    │
│    - **Duration:** 3 hours                                                                                      │
│    - **Meeting Point:** Louvre Pyramid                                                                          │
│    - **Cost:** $85 per person                                                                                   │
│                                                                                                                 │
│  **Flow:** Start the day with a breathtaking view from 


=== Delegator Task Complete ===



╭─────────────────────────────────────────── Tracing Preference Saved ────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing has been disabled.                                                                               │
│                                                                                                                 │
│  Your preference has been saved. Future Crew/Flow executions will not collect traces.                           │
│                                                                                                                 │
│  To enable tracing later, do any one of these:                                                                  │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [19]:
display(HTML('<div style="background-color: #000; padding: 10px; border-radius: 5px; border: 1px solid #d3d3d3;"></hr><h2>🔽 &nbsp; Full travel plan – Report</h2></hr></div>'))

for task in itinerary.tasks_output:      
    display(Markdown(task.raw))

After reviewing the available flight options from New York to Paris on May 7, 2025, here are the best choices based on Alex Johnson's preferences for morning departures, cost, and duration:

1. **Air France**
   - **Departure:** New York (JFK) at 10:30 AM
   - **Arrival:** Paris (CDG) at 11:00 PM
   - **Duration:** 7h 30m
   - **Price:** $850
   - **Notes:** This flight meets the preference for a morning departure and offers a relatively short duration at a reasonable price.

2. **Delta Airlines**
   - **Departure:** New York (JFK) at 5:30 PM
   - **Arrival:** Paris (CDG) at 6:15 AM (next day)
   - **Duration:** 7h 45m
   - **Price:** $780
   - **Notes:** While this flight is less expensive, it departs in the evening, which does not align with the preference for a morning departure.

3. **United Airlines**
   - **Departure:** New York (EWR) at 8:45 PM
   - **Arrival:** Paris (CDG) at 9:40 AM (next day)
   - **Duration:** 7h 55m
   - **Price:** $920
   - **Notes:** This flight is the most expensive and departs at night, which is not ideal for the morning departure preference.

**Recommendation:**

The best option is the **Air France** flight. It aligns with the desired morning departure, offers a competitive duration, and the price is reasonable compared to the other options. This selection ensures a balance between convenience and cost-effectiveness.

Alex should book the **Air France** flight for his anniversary trip to ensure it matches his morning departure preference with a reasonable travel time and cost.

After reviewing the available hotel options in Paris from May 7 to May 14, 2025, here are the choices and my recommended selection for Alex Johnson's trip:

### Available Hotel Options:

1. **Paris Marriott Champs Elysees**
   - **Price:** $450 per night
   - **Location:** Central Paris
   - **Rating:** 4.5
   - **Amenities:** Spa, Restaurant, Room Service

2. **Citadines Saint-Germain-des-Prés**
   - **Price:** $320 per night
   - **Location:** Saint-Germain
   - **Rating:** 4.2
   - **Amenities:** Kitchenette, Laundry, Wifi

3. **Ibis Paris Eiffel Tower**
   - **Price:** $380 per night
   - **Location:** Near Eiffel Tower
   - **Rating:** 4.0
   - **Amenities:** Restaurant, Bar, Wifi

### Recommendation:

The best hotel option for Alex Johnson's anniversary trip is **Citadines Saint-Germain-des-Prés**. Here's why:

- **Budget Alignment:** At $320 per night, this hotel is well within the budget limit of $400 per night, allowing a comfortable stay while keeping costs manageable compared to the other options.
- **Location & Accessibility:** Located in the Saint-Germain area, it offers a central location in Paris, known for its vibrant cultural and historic ambiance. It's convenient for accessing various city attractions through public transport.
- **Amenities:** The offer includes amenities such as a kitchenette and WiFi. Having a kitchenette can add value by allowing flexibility in meal planning and possibly reducing dining costs.

This choice balances affordability with a high level of comfort and convenience, fitting ideally with the trip’s budget and preferences for a central stay in Paris. 

**Booking-Ready Format:**
- **Hotel Name:** Citadines Saint-Germain-des-Prés
- **Check-in:** May 7, 2025
- **Check-out:** May 14, 2025
- **Price per Night:** $320
- **Total Stay Price:** $2,240
- **Location:** Saint-Germain, Central Paris
- **Rating:** 4.2
- **Amenities:** Kitchenette, Laundry, Wifi

Alex should consider booking this accommodation to enjoy a memorable and comfortable anniversary trip in Paris.

Based on the transportation options identified for Alex Johnson's anniversary trip to Paris, here is a detailed transportation plan covering each necessary transfer, aligned with preferences for cost, convenience, and comfort:

### **Airport to Hotel Transfer (CDG to Citadines Saint-Germain-des-Prés)**
**Recommended Option: Taxi**
- **Type:** Taxi
- **Cost:** $22.5
- **Duration:** 20 minutes
- **Frequency:** On demand
- **Pros:** Offers door-to-door service and comfort, which is ideal upon arrival to ensure convenience and ease after a long flight.
- **Cons:** Subject to traffic, but the arrival time around 11:00 PM should generally see lighter traffic.

**Reasoning:** Given the time of arrival at 11:00 PM, a taxi provides a comfortable and direct option after the long flight. It ensures Alex and his travel companion can swiftly and conveniently reach the hotel without navigating public transport with luggage late at night.

### **Transportation Between Daily Activities (Hotel to Eiffel Tower)**
**Recommended Option: Metro**
- **Type:** Metro
- **Cost:** $1.9
- **Duration:** 25 minutes
- **Frequency:** Every 5 minutes
- **Route:** Line 1 to Châtelet, then Line 4 to the destination.
- **Pros:** Fast, avoids surface traffic, and is cost-effective with frequent service that fits into the budget and allows spontaneous travel.
- **Cons:** Can be crowded during peak hours, but planning around busy times should mitigate this.

**Reasoning:** The metro provides a fast and economical mode of transport for daily sightseeing. Its reliability and frequency match the need for flexibility and efficient use of time while exploring Paris.

### **Hotel to Airport Transfer (Citadines Saint-Germain-des-Prés to CDG)**
**Recommended Option: Taxi**
- **Type:** Taxi
- **Cost:** $22.5
- **Duration:** 20 minutes
- **Frequency:** On demand
- **Pros:** Ensures punctuality and ease of departure with direct service from the hotel to the airport.
- **Cons:** More expensive than public transport but offers stress-free travel especially when leaving for flights.

**Reasoning:** For the return trip to the airport, a taxi is recommended as it provides reliability and ensures that Alex makes the return flight on time without additional stress, particularly given the potential complexities of traveling on public transit with luggage.

### **Transportation Pass Options**
**Recommendation:** Day Pass / Paris Visite
- **Day Pass Cost:** $7.5
  - **Valid For:** Unlimited travel for 24 hours
  - **Recommended If:** Making more than 4 trips in a day.
  
- **Paris Visite Cost:** $12.0
  - **Valid For:** Unlimited travel for 1 day, includes discounts to attractions
  - **Recommended If:** Planning to visit multiple tourist sites in a single day.

**Reasoning:** Depending on the day’s activities, opting for the Paris Visite pass can provide flexibility and added benefits, especially when planning to visit multiple attractions which also aligns with the traveler's interest in cultural activities and is cost-efficient for days with multiple stops.

This transportation plan effectively balances cost, convenience, and comfort while ensuring Alex Johnson's anniversary trip is as seamless and enjoyable as possible.

Here's a carefully curated day-by-day activity plan for Alex Johnson's anniversary trip to Paris. The itinerary includes cultural and relaxing experiences suited for a moderate pace, matching their interests:

### Day 1: Arrival and Evening Cruise (May 7, 2025)
- **Activity:** Seine River Dinner Cruise
  - **Description:** Evening cruise along the Seine with a 3-course French dinner and wine.
  - **Start Time:** 7:30 PM
  - **Duration:** 2.5 hours
  - **Meeting Point:** Port de la Bourdonnais
  - **Cost:** $120 per person

**Flow:** After arriving and settling into the hotel, this relaxing dinner cruise will provide a delightful introduction to Paris and its iconic landmarks beautifully lit at night.

### Day 2: Iconic Paris Landmarks (May 8, 2025)
- **Morning Activity:** Eiffel Tower Skip-the-Line Tour
  - **Description:** Priority access to the Eiffel Tower with a guided tour of the 1st and 2nd floors.
  - **Start Time:** 10:00 AM
  - **Duration:** 2 hours
  - **Meeting Point:** Eiffel Tower South Entrance
  - **Cost:** $65 per person

- **Afternoon Activity:** Louvre Museum Guided Tour
  - **Description:** Expert-guided tour of the Louvre's masterpieces including Mona Lisa.
  - **Start Time:** 2:00 PM
  - **Duration:** 3 hours
  - **Meeting Point:** Louvre Pyramid
  - **Cost:** $85 per person

**Flow:** Start the day with a breathtaking view from the Eiffel Tower and continue to the artistic treasures of the Louvre, providing a blend of iconic experiences.

### Day 3: Leisurely Strolls and Gardens (May 9, 2025)
- **Flexible Activity:** Explore Tuileries Garden and Luxembourg Gardens
  - **Description:** Take leisurely strolls through two of Paris's most beautiful parks.
  - **Cost:** Free

- **Evening:** Relax and indulge in a quiet dinner at one of the local bistros.

**Flow:** Enjoy a leisurely day appreciating the natural beauty of Paris's renowned gardens at your own pace.

### Day 4: Cultural Immersion (May 10, 2025)
- **Activity:** Musée d'Orsay Visit
  - **Description:** Explore an extensive collection of Impressionist and Post-Impressionist masterpieces.
  - **Duration:** Flexible
  - **Cost:** To be checked on-site

**Flow:** Spend the day at Musée d'Orsay, immersed in rich art, continuing the cultural theme and providing ample relaxation time.

### Day 5: Exploration and Romance (May 11, 2025)
- **Activity:** Montmartre Walking Tour
  - **Description:** Discover the artistic heart of Paris, visiting iconic spots like Sacré-Cœur.
  - **Cost:** To be checked on-site or enjoy a self-guided tour.

- **Evening:** Romantic dinner with view opportunities overlooking the city.

**Flow:** Explore the cobbled streets of Montmartre, combining history, art, and romance in this quaint part of Paris.

### Day 6: Relaxation and Culinary Delights (May 12, 2025)
- **Activity:** Spa Experience at Hotel or Nearby
  - **Description:** Enjoy a relaxing spa treatment to unwind together.
  - **Cost:** To be arranged directly (optional, based on preference)

**Flow:** Focus on relaxation with a day designed for rejuvenation and easy rest, ending with a memorable dining experience.

### Day 7: Last Glimpse and Departure (May 13, 2025)
- **Activity:** Leisurely Breakfast and Use of Paris Visite for Quick Last Stops
  - **Description:** Use the Paris Visite pass for any last-minute attractions or leisurely exploration.
  - **Cost:** $12 per person for the pass

**Flow:** Pack in any last moments of Paris magic with flexibility, preparing for departure with reflective joy.

---

**Note:** This itinerary has been designed with consideration for a moderate pace, ensuring that Alex and his companion can savor each experience without feeling rushed or overwhelmed, aligning with the ambiance of a romantic, cultural anniversary trip.